# Verificación del entorno

**Semana 1** · ambos niveles · 5 minutos · CPU o GPU

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendiento_IA/blob/main/verificar_entorno.ipynb)

Este notebook comprueba que tu entorno tiene todo lo que usaremos durante el año: las librerías con las versiones del curso y, si hay, la GPU. Ejecútalo completo (*Entorno de ejecución → Ejecutar todas* en Colab) y revisa el resumen del final.

Las versiones del curso están en [`requirements.txt`](https://github.com/melvinpqbsc/Alto_Rendiento_IA/blob/main/requirements.txt) y son las mismas que trae Google Colab.

## 1. Python

In [ ]:
import sys
import platform

EN_COLAB = "google.colab" in sys.modules
print("Python:", platform.python_version())
print("Sistema:", platform.system(), platform.machine())
print("Entorno:", "Google Colab" if EN_COLAB else "local")

if sys.version_info[:2] != (3, 13):
    print("⚠ El curso usa Python 3.13. Con otra versión algunas librerías pueden comportarse distinto.")

## 2. Librerías

Compara lo que tienes instalado con [`requirements.txt`](https://github.com/melvinpqbsc/Alto_Rendiento_IA/blob/main/requirements.txt). Si ejecutas el notebook desde tu copia local del repositorio, usa ese archivo; si no (por ejemplo, en Colab), lo descarga de GitHub.

In [ ]:
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path
from urllib.request import urlopen

URL_REQUIREMENTS = "https://raw.githubusercontent.com/melvinpqbsc/Alto_Rendiento_IA/main/requirements.txt"

archivo_local = Path("requirements.txt")
if archivo_local.exists():
    texto = archivo_local.read_text(encoding="utf-8")
else:
    texto = urlopen(URL_REQUIREMENTS).read().decode("utf-8")

# Cada línea con "paquete==versión"; lo que va después de "#" es un comentario.
# Las secciones empiezan con "# ---". Dos son especiales:
#   "solo local": Colab trae sus propias versiones, así que en Colab no se revisan.
#   "No vienen en Colab": cada notebook que las usa las instala en su primera celda.
esperadas = {}
opcionales = set()
seccion = ""
for linea in texto.splitlines():
    if linea.startswith("# ---"):
        seccion = linea
        continue
    linea = linea.split("#")[0].strip()
    if "==" not in linea or (EN_COLAB and "solo local" in seccion):
        continue
    nombre, v = linea.split("==")
    esperadas[nombre] = v
    if "No vienen en Colab" in seccion:
        opcionales.add(nombre)

faltan, distintas = [], []
print(f"{'paquete':<24}{'curso':<14}{'instalada':<22}")
print("-" * 62)
for nombre, v_curso in esperadas.items():
    try:
        v_inst = version(nombre)
    except PackageNotFoundError:
        v_inst = None
    # torch en Colab o con CUDA reporta "2.11.0+cu130": comparamos sin el sufijo
    if v_inst is None:
        marca = "· se instala cuando se use" if nombre in opcionales else "✗ FALTA"
        if nombre not in opcionales:
            faltan.append(nombre)
    elif v_inst.split("+")[0] == v_curso:
        marca = "✓"
    else:
        marca = "≠"
        distintas.append(nombre)
    print(f"{nombre:<24}{v_curso:<14}{(v_inst or '—'):<22}{marca}")

## 3. GPU

En Colab, activa la GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*. La mayor parte del curso funciona sin GPU; desde el Bloque 2 los entrenamientos son mucho más rápidos con ella.

In [ ]:
import torch

if torch.cuda.is_available():
    dispositivo = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    dispositivo = torch.device("mps")  # Mac con chip Apple
    print("GPU: Apple (MPS)")
else:
    dispositivo = torch.device("cpu")
    print("Sin GPU: se usará la CPU.")

## 4. Prueba rápida

Entrena una regresión lineal con PyTorch en el dispositivo elegido. Si la pérdida baja, todo funciona.

In [ ]:
torch.manual_seed(0)

x = torch.randn(1000, 3, device=dispositivo)
w_real = torch.tensor([2.0, -1.0, 0.5], device=dispositivo)
y = x @ w_real + 0.1 * torch.randn(1000, device=dispositivo)

modelo = torch.nn.Linear(3, 1).to(dispositivo)
optimizador = torch.optim.SGD(modelo.parameters(), lr=0.1)

for paso in range(100):
    perdida = torch.nn.functional.mse_loss(modelo(x).squeeze(1), y)
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()

print(f"Pérdida final: {perdida.item():.4f}")
print("Pesos aprendidos:", modelo.weight.data.squeeze().cpu().numpy().round(2), "(los reales: [2, -1, 0.5])")
assert perdida.item() < 0.05, "La pérdida no bajó: algo anda mal con PyTorch."
entrenamiento_ok = True

## Resumen

In [ ]:
if faltan:
    print("✗ Faltan librerías:", ", ".join(faltan))
    print("  En tu computadora: uv pip install -r requirements.txt")
if distintas:
    print("≠ Versiones distintas a las del curso:", ", ".join(distintas))
    print("  En Colab es normal que cambien durante el año; avísale al profesor si algo falla.")
    print("  En tu computadora: uv pip install -r requirements.txt")
print("GPU:", "sí" if dispositivo.type != "cpu" else "no")
if not faltan and entrenamiento_ok:
    print("\n✓ Tu entorno está listo para el curso.")

Si algo falla, copia el mensaje de error y sigue el orden de siempre: la documentación, dos compañeros y después el profesor.